# 00. Chuẩn bị và kiểm tra

Chạy notebook này một lần trước khi train. Nó xác nhận GPU, secret, dữ liệu và model, rồi đo tốc độ thật để thay các con số ước tính trong kế hoạch.

**Yêu cầu:** bật GPU và thêm secret `HF_TOKEN`, `WANDB_API_KEY` theo hướng dẫn trong README.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Đổi thành địa chỉ repo của bạn. Repo private thì thêm secret GITHUB_TOKEN trên Colab.
REPO_URL = "https://github.com/TEN-TAI-KHOAN/vihsd-lora.git"

def github_token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    if "TEN-TAI-KHOAN" in REPO_URL:
        raise ValueError("Hãy sửa REPO_URL thành địa chỉ repo GitHub của bạn rồi chạy lại ô này.")
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    token = github_token()
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=False)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, free_space_gb, load_secrets

print("Nền tảng:", detect_platform())
print("GPU:", describe_gpu())
print("Secret đã có:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
free_gb = free_space_gb(settings.output_root)
print("Kết quả lưu tại:", settings.output_root)
print(f"Dung lượng còn trống: {free_gb} GB")
if free_gb < 6:
    print("Cảnh báo: còn dưới 6GB. Lượt full fine-tune cần khoảng 3GB, hãy dọn Drive và làm trống thùng rác.")

## Kiểm tra dữ liệu

Ô dưới dừng lại nếu số mẫu từng nhãn không khớp bản ViHSD chuẩn.

In [ ]:
import pandas as pd
from vihsd_lora.data import label_counts, length_report, load_vihsd
from vihsd_lora.metrics import majority_baseline

dataset = load_vihsd(settings)
display(pd.DataFrame(label_counts(dataset)).T)

baseline = majority_baseline(dataset["test"]["labels"])
print(f"Luôn đoán CLEAN trên test: Acc {baseline['accuracy']:.2f}, macro F1 {baseline['macro_f1']:.2f}")

## Kiểm tra tokenizer và độ dài bình luận

`truncated_pct` là tỉ lệ bình luận dài hơn `max_length` và bị cắt.

In [ ]:
from vihsd_lora.modeling import load_tokenizer

tokenizer = load_tokenizer(settings.model_name)
print(length_report(dataset["train"], tokenizer, settings.max_length))

## Kiểm tra số tham số

Số mong đợi với ViSoBERT: LoRA r=8 train 887,811 tham số, full fine-tune train 97,568,259 tham số.

In [ ]:
from vihsd_lora.config import RUNS
from vihsd_lora.modeling import build_model, count_parameters

rows = []
for key, run in RUNS.items():
    if key == "E":
        continue
    rows.append({"lượt": key, "mô tả": run.describe(), **count_parameters(build_model(run, settings))})
display(pd.DataFrame(rows))

## Đo tốc độ thật

Chạy 200 step cho LoRA và cho full fine-tune. Bảng cuối là thời gian train ước tính theo tốc độ đo được, chưa tính thời gian chấm dev.

In [ ]:
from vihsd_lora.speedtest import estimate_hours, measure

SPEED_STEPS = 200
speed_lora = measure("A", steps=SPEED_STEPS, settings=settings)
speed_full = measure("B", steps=SPEED_STEPS, settings=settings)
print(speed_lora)
print(speed_full)
display(estimate_hours({"lora": speed_lora["seconds_per_step"], "full": speed_full["seconds_per_step"]}, settings))